# Assignment 1 · The evaluation engine
**Quantitative Asset Management · Frankfurt School · 2026 Group3**

**AI coding assistants used:** OpenAI ChatGPT (Codex) and Anthropic Claude (Claude Code).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from QAM26_shrinkage import shrinkage
from QAM26_optimization import mean_variance_long_only

GAMMA, COST = 4, 0.001
START, END = "1980-01", "2026-09"


def load(filename):
    data = pd.read_csv(f"QAM26_data/{filename}", index_col="month") / 100
    data.index = pd.PeriodIndex(data.index, freq="M")
    return data


industries = load("industries30_returns.csv")
factors = load("factors.csv")
rf = factors["RF"]
excess = industries.sub(rf, axis=0)

## Part A

### A1. The engine
Row $t$ of the weights table holds $\theta_{t-1}$, chosen from months $t-120,\dots,t-1$; $r_{p,t}=\theta_{t-1}'r_t$. Turnover compares the drifted weights with the next row; the trade at the end of month $t$ is charged to month $t+1$.

In [2]:
def performance(r, gamma=GAMMA):
    m, s = 12 * r.mean(), np.sqrt(12) * r.std(ddof=1)
    return {"Mean (%)": 100 * m, "Volatility (%)": 100 * s, "Sharpe": m / s, "CE (%)": 100 * (m - gamma / 2 * s**2)}


def run_backtest(returns, rf, rule, window=120, start="1980-01", end="2026-09", cost=0.0, shift=0, gamma=GAMMA):
    months = returns.index
    first, last = months.get_loc(pd.Period(start, "M")), months.get_loc(pd.Period(end, "M"))
    if first < window:
        raise ValueError(f"returns has only {first} months before {start}, the window needs {window}.")
    x = returns.sub(rf, axis=0)
    period = months[first:last + 1]
    weights = pd.DataFrame([rule(x.iloc[t - window + shift:t + shift]) for t in range(first, last + 1)],
                           index=period, columns=returns.columns)
    rp = (weights * x.loc[period]).sum(axis=1)                                   # before costs
    drift = weights.mul(1 + returns.loc[period]).div(1 + rf.loc[period] + rp, axis=0)
    turnover = (weights.shift(-1) - drift).abs().sum(axis=1).iloc[:-1]          # 560 month-ends
    net = rp.copy()
    net.iloc[1:] -= cost * turnover.to_numpy()                                   # paid in t+1
    measures = performance(net, gamma)
    measures["Turnover"] = turnover.mean()
    measures["Gross exposure"] = weights.abs().sum(axis=1).mean()
    return {"returns": net, "weights": weights, "measures": measures}

### A2. Six rules

In [3]:
def moments(w):
    x = np.asarray(w, dtype=float)
    return x.mean(axis=0), np.cov(x, rowvar=False)          # sample moments, divisor T-1


def rule_equal_weights(w):
    return np.ones(w.shape[1]) / w.shape[1]


def rule_inverse_volatility(w):
    inv = 1 / w.std(ddof=1).to_numpy()
    return inv / inv.sum()


def rule_plugin(w, gamma=GAMMA):
    mu, Sigma = moments(w)
    return np.linalg.solve(Sigma, mu) / gamma


def rule_shrinkage(w):
    mu, Sigma, delta = shrinkage(w.to_numpy(), 1)
    return np.linalg.solve(Sigma, mu) / GAMMA


def rule_long_only(w):
    return mean_variance_long_only(*moments(w), GAMMA)


def rule_minimum_variance(w):
    z = np.linalg.solve(moments(w)[1], np.ones(w.shape[1]))
    return z / z.sum()


rules = {"1. 1/N": rule_equal_weights, "2. Inverse volatility": rule_inverse_volatility,
         "3. Plug-in MV": rule_plugin, "4. Shrinkage MV": rule_shrinkage,
         "5. Long-only MV": rule_long_only, "6. Minimum variance": rule_minimum_variance}
names = list(rules)
gross = {n: run_backtest(industries, rf, r) for n, r in rules.items()}
net = {n: run_backtest(industries, rf, r, cost=COST) for n, r in rules.items()}

### A3. Check values

In [4]:
summary = pd.DataFrame({n: gross[n]["measures"] for n in names}).T
check = summary.loc[names[:2], ["Mean (%)", "Volatility (%)", "Sharpe", "Turnover"]]
display(check.round(3))
np.testing.assert_allclose(check.round({"Mean (%)": 2, "Volatility (%)": 2, "Sharpe": 2, "Turnover": 3}),
                           [[9.04, 16.27, 0.56, 0.031], [8.89, 15.50, 0.57, 0.029]])

# Rules 3-6 (no check values): defining conditions in the first window, 1970/01-1979/12.
first = excess.loc["1970-01":"1979-12"]
mu, Sigma = moments(first)
w3, w4, w5, w6 = (gross[n]["weights"].iloc[0].to_numpy() for n in names[2:])
mu_s, Sigma_s, _ = shrinkage(first.to_numpy(), 1)
assert np.allclose(GAMMA * Sigma @ w3, mu)                       # rule 3: gamma Sigma theta = mu
assert np.allclose(GAMMA * Sigma_s @ w4, mu_s)                   # rule 4: same, shrinkage moments
assert np.isclose(w5.sum(), 1) and (w5 >= -1e-12).all()          # rule 5: fully invested, no shorts
assert np.isclose(w6.sum(), 1) and np.allclose(Sigma @ w6, w6 @ Sigma @ w6)   # rule 6: Sigma theta ∝ 1
print("Rules 3-6 satisfy their defining conditions.")

,Mean (%),Volatility (%),Sharpe,Turnover
1. 1/N,9.036,16.271,0.555,0.031
2. Inverse volatility,8.886,15.497,0.573,0.029


Rules 3-6 satisfy their defining conditions.


### A4. The table (mean, volatility, CE in % per year; last three columns net of costs)

In [5]:
table = summary.copy()
for col in ["Mean (%)", "Sharpe", "CE (%)"]:
    table["Net " + col] = [net[n]["measures"][col] for n in names]
table.loc["Market (Mkt-RF)"] = pd.Series(performance(factors.loc[START:END, "Mkt-RF"]))
display(table.round(3))

,Mean (%),Volatility (%),Sharpe,CE (%),Turnover,Gross exposure,Net Mean (%),Net Sharpe,Net CE (%)
1. 1/N,9.036,16.271,0.555,3.741,0.031,1.000,8.999,0.553,3.704
2. Inverse volatility,8.886,15.497,0.573,4.083,0.029,1.000,8.851,0.571,4.048
3. Plug-in MV,28.928,86.002,0.336,-119.000,9.189,30.448,17.922,0.208,-130.570
4. Shrinkage MV,20.835,34.690,0.601,-3.232,1.129,6.535,19.483,0.562,-4.544
5. Long-only MV,10.407,19.804,0.526,2.563,0.198,1.000,10.171,0.514,2.326
6. Minimum variance,7.595,13.024,0.583,4.202,0.471,4.122,7.031,0.540,3.641
Market (Mkt-RF),8.856,15.549,0.570,4.021,NaN,NaN,NaN,NaN,NaN


### A5. The in-sample Sharpe ratio

In [6]:
mu_is, Sigma_is = moments(excess.loc[START:END])
sr_in_sample = np.sqrt(12 * mu_is @ np.linalg.solve(Sigma_is, mu_is))
print(f"In-sample SR: {sr_in_sample:.3f};  out-of-sample SR of rule 3: {table.loc[names[2], 'Sharpe']:.3f}")

In-sample SR: 1.072;  out-of-sample SR of rule 3: 0.336


### A6. Questions

**(a)** The in-sample Sharpe ratio is **1.072**, the out-of-sample Sharpe ratio of rule 3 **0.336**. The in-sample weights are chosen with the moments of the same 561 months on which they are evaluated, so estimation noise that happened to pay off counts as skill: it is the maximum ex-post Sharpe ratio and biased upward. Rule 3 only knows the previous 120 months; its noisy $\hat\mu$, amplified by $\hat\Sigma^{-1}$, gives extreme positions that do not pay off. Only the **0.336** could have been earned: the 1.072 requires knowing in 1980 the returns of 1980–2026.

**(b)** The standard error of one Sharpe ratio is $\sqrt{12/561}\approx$ **0.146**, of a difference about **0.207** if returns are uncorrelated. Rules 1, 2, 4, 5 and 6 lie between 0.526 and 0.601: the largest gap, 0.075, is about a third of a standard error, so **they cannot be ranked**. Rule 3 (0.336) lies 0.19–0.26 below the others, about one standard error of a difference (somewhat more with positive correlation): weak evidence that it is worse, no more. What sets rule 3 apart are the other columns: volatility **86.00%**, CE **−119.00%**, turnover **9.19** and gross exposure **30.45**. Plug-in mean-variance turns estimation error into huge, unstable, leveraged positions; a $\gamma=4$ investor would rather hold T-bills.

**(c)** Before costs: 4 (0.601) > 6 (0.583) > 2 (0.573) > 1 (0.555) > 5 (0.526) > 3 (0.336). Net of costs: 2 (0.571) > 4 (0.562) > 1 (0.553) > 6 (0.540) > 5 (0.514) > 3 (0.208). Inverse volatility moves from 3rd to 1st and 1/N from 4th to 3rd; shrinkage drops from 1st to 2nd and minimum variance from 2nd to 4th. The **turnover** column explains it: rules 1 and 2 trade about 0.03 of wealth per month, rules 4 and 6 1.13 and 0.47. These changes are much smaller than the standard errors in (b).

**(d) Prediction.** Rule 3 holds $\frac1\gamma\hat\Sigma^{-1}\hat\mu$, so a higher $\gamma$ only scales all weights and the excess return by $4/\gamma$. Mean and volatility scale by the same factor: the **Sharpe ratio does not change**. The CE becomes $CE(\gamma)=\frac4\gamma m_4-\frac\gamma2\left(\frac4\gamma\right)^2 s_4^2=\frac4\gamma\,CE(4)$: it moves toward zero but keeps its sign, about −79% for $\gamma=6$ and −60% for $\gamma=8$.

In [7]:
display(pd.DataFrame({f"gamma = {g}": run_backtest(industries, rf, lambda w, g=g: rule_plugin(w, g), gamma=g)["measures"]
                      for g in (4, 6, 8)}).T[["Sharpe", "CE (%)"]].round(3))

,Sharpe,CE (%)
gamma = 4,0.336,-119.000
gamma = 6,0.336,-79.334
gamma = 8,0.336,-59.500


*Result.* As predicted: Sharpe ratio 0.336 for all three, CE −119.00%, −79.33% and −59.50%. **No $\gamma$ makes the CE positive:** with $a$ and $b$ the mean and volatility of the unscaled portfolio $\hat\Sigma^{-1}\hat\mu$, $CE=(a-b^2/2)/\gamma$. $\gamma$ changes only the size, never the sign, and here $a-b^2/2=4\,CE(4)<0$. Equivalently, $CE>0$ would require $SR>\gamma s/2=2s_4=1.72$, while the realised Sharpe ratio is 0.336.

## Part B

### B1. One month of look-ahead

In [8]:
display(pd.DataFrame({n: {"shift = 0": table.loc[n, "Sharpe"],
                          "shift = 1": run_backtest(industries, rf, rules[n], shift=1)["measures"]["Sharpe"]}
                      for n in (names[0], names[2], names[3])}).T.round(3))

,shift = 0,shift = 1
1. 1/N,0.555,0.555
3. Plug-in MV,0.336,2.337
4. Shrinkage MV,0.601,1.245


1/N does not use the data, so it stays at 0.555. Rule 3 rises from 0.336 to **2.337**, rule 4 from 0.601 to 1.245. With `shift=1`, $\hat\mu$ contains $r_t/120$, so the weights contain $\frac{1}{120\gamma}\hat\Sigma^{-1}r_t$, which adds $\frac{1}{120\gamma}r_t'\hat\Sigma^{-1}r_t>0$ to the return of month $t$, **in every month**. $\hat\Sigma^{-1}$ blows this small look-ahead up, the same mechanism that gives rule 3 a gross exposure of 30. Shrinkage pulls the means together and stabilises the covariance matrix, so rule 4 gains less.

### B2. The length of the window, in the data

In [9]:
windows = (60, 120, 240)
b2 = pd.DataFrame({n: [run_backtest(industries, rf, rules[n], window=w)["measures"]["Sharpe"] for w in windows]
                   for n in names[2:4]}, index=pd.Index(windows, name="Window"))
display(b2.round(3))

,3. Plug-in MV,4. Shrinkage MV
Window,,
60,0.265,0.477
120,0.336,0.601
240,0.179,0.505


### B3. The length of the window, in a world that does not change

In [10]:
calibration = excess.loc["1960-01":END]
mu_true, Sigma_true = moments(calibration)
theta_star = np.linalg.solve(Sigma_true, mu_true) / GAMMA
print(f"SR* = {np.sqrt(12 * mu_true @ np.linalg.solve(Sigma_true, mu_true)):.3f}")

SEED = 20261123
rng = np.random.default_rng(SEED)
zero_rf = pd.Series(0.0, index=calibration.index)
cases = [("1. 1/N", rule_equal_weights, 120), ("theta*", lambda w: theta_star, 120)] + \
        [(f"{n}, {w}", rules[n], w) for n in names[2:4] for w in windows]
sim = []
for _ in range(100):
    history = pd.DataFrame(rng.multivariate_normal(mu_true, Sigma_true, size=801),
                           index=calibration.index, columns=industries.columns)
    sim.append({label: run_backtest(history, zero_rf, rule, window=w)["measures"]["Sharpe"]
                for label, rule, w in cases})
sim = pd.DataFrame(sim)
print(f"Seed {SEED}, 100 histories")
display(pd.DataFrame({"Average Sharpe ratio": sim.mean(), "SD across histories": sim.std(ddof=1)}).round(3))
print("Rule 3: SR(240) < SR(120) in", (sim["3. Plug-in MV, 240"] < sim["3. Plug-in MV, 120"]).sum(), "of 100 histories")

SR* = 0.890


Seed 20261123, 100 histories


,Average Sharpe ratio,SD across histories
1. 1/N,0.460,0.155
theta*,0.885,0.174
"3. Plug-in MV, 60",0.186,0.161
"3. Plug-in MV, 120",0.317,0.179
"3. Plug-in MV, 240",0.472,0.175
"4. Shrinkage MV, 60",0.315,0.186
"4. Shrinkage MV, 120",0.414,0.179
"4. Shrinkage MV, 240",0.535,0.164


Rule 3: SR(240) < SR(120) in 9 of 100 histories


**(a)** In the simulated world both rules improve with the window: rule 3 0.186 / 0.317 / 0.472 and rule 4 0.315 / 0.414 / 0.535 for 60 / 120 / 240 months. The parameters never change, so old months are as informative as new ones and a longer window only reduces estimation error. In the data (B2) the pattern is hump-shaped: rule 3 0.265 / 0.336 / 0.179, rule 4 0.477 / 0.601 / 0.505. From 60 to 120 months less estimation error helps; from 120 to 240 the older data describe a different world, because means and covariances change over time. The difference between the two patterns is this non-stationarity, which the simulation removes by construction, plus the luck of observing only one history.

**(b)** In **9 of 100** histories rule 3 has a lower Sharpe ratio with 240 months than with 120. In a world that does not change, a longer window almost always helps, so a reversal like the one in B2 (0.336 → 0.179) is rare. B2 is therefore evidence that the real parameters change over time, but not proof: the data are a single history.

**(c)** **By less.** Each sample mean is the true mean plus estimation error, so the sample means differ from each other by more than the true means do. The simulated world therefore builds in larger differences in expected returns than the real world has, which favours rules 3 and 4. Outside the simulation 1/N should do relatively better. Even inside it, rules 3 and 4 with 120 months fall short of 1/N (0.317 and 0.414 vs 0.460); in the data rule 3 is far below 1/N (0.336 vs 0.555).